# Imports and file loading

In [2]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path(".")

business_path = DATA_DIR / "yelp_academic_dataset_business.json"
review_path = DATA_DIR / "yelp_academic_dataset_review.json"
checkin_path = DATA_DIR / "yelp_academic_dataset_checkin.json"
tip_path = DATA_DIR / "yelp_academic_dataset_tip.json"
user_path = DATA_DIR / "yelp_academic_dataset_user.json"

# Filter Arizona coffee-related businesses

In [ ]:
business = pd.read_json(business_path, lines=True)

coffee_az = business[
    (business["state"] == "AZ") &
    (business["categories"].fillna("").str.contains("Coffee|Cafe|Tea", case=False, regex=True))
].copy()

coffee_az = coffee_az[[
    "business_id", "name", "city", "state", "latitude", "longitude",
    "stars", "review_count", "categories", "is_open"
]]

coffee_business_ids = set(coffee_az["business_id"])

print("Filtered businesses:", coffee_az.shape)
coffee_az.head()

## Filter reviews linked to selected businesses

In [ ]:
review_chunks = []

for chunk in pd.read_json(review_path, lines=True, chunksize=50000):
    filtered = chunk[chunk["business_id"].isin(coffee_business_ids)].copy()
    if not filtered.empty:
        review_chunks.append(filtered)

coffee_reviews = pd.concat(review_chunks, ignore_index=True)

coffee_reviews = coffee_reviews[[
    "review_id", "user_id", "business_id", "stars", "useful",
    "funny", "cool", "text", "date"
]]

coffee_reviews["date"] = pd.to_datetime(coffee_reviews["date"], errors="coerce")
coffee_reviews["review_length"] = coffee_reviews["text"].fillna("").str.len()

print("Filtered reviews:", coffee_reviews.shape)
coffee_reviews.head()

## Filter tips linked to selected businesses

In [ ]:
tip_chunks = []

for chunk in pd.read_json(tip_path, lines=True, chunksize=50000):
    filtered = chunk[chunk["business_id"].isin(coffee_business_ids)].copy()
    if not filtered.empty:
        tip_chunks.append(filtered)

coffee_tips = pd.concat(tip_chunks, ignore_index=True)

coffee_tips = coffee_tips[[
    "user_id", "business_id", "text", "date", "compliment_count"
]]

coffee_tips["date"] = pd.to_datetime(coffee_tips["date"], errors="coerce")
coffee_tips["tip_length"] = coffee_tips["text"].fillna("").str.len()

print("Filtered tips:", coffee_tips.shape)
coffee_tips.head()

## Filter check-ins linked to selected businesses

In [ ]:
checkins = pd.read_json(checkin_path, lines=True)

coffee_checkins = checkins[checkins["business_id"].isin(coffee_business_ids)].copy()

coffee_checkins["checkin_dates"] = coffee_checkins["date"].fillna("")
coffee_checkins["total_checkins"] = coffee_checkins["checkin_dates"].apply(
    lambda x: 0 if not x else len([d for d in str(x).split(",") if d.strip()])
)

print("Filtered check-ins:", coffee_checkins.shape)
coffee_checkins.head()

## Filter users linked to selected reviews and tips

In [ ]:
coffee_user_ids = set(coffee_reviews["user_id"].dropna()) | set(coffee_tips["user_id"].dropna())

user_chunks = []

for chunk in pd.read_json(user_path, lines=True, chunksize=50000):
    filtered = chunk[chunk["user_id"].isin(coffee_user_ids)].copy()
    if not filtered.empty:
        user_chunks.append(filtered)

coffee_users = pd.concat(user_chunks, ignore_index=True)

coffee_users = coffee_users[[
    "user_id", "review_count", "yelping_since", "useful", "funny",
    "cool", "fans", "average_stars"
]]

print("Filtered users:", coffee_users.shape)
coffee_users.head()

# Save analysis-ready CSV files

In [ ]:
coffee_az.to_csv(DATA_DIR / "az_coffee_business.csv", index=False)
coffee_reviews.to_csv(DATA_DIR / "az_coffee_reviews.csv", index=False)
coffee_tips.to_csv(DATA_DIR / "az_coffee_tips.csv", index=False)
coffee_checkins.to_csv(DATA_DIR / "az_coffee_checkins.csv", index=False)
coffee_users.to_csv(DATA_DIR / "az_coffee_users.csv", index=False)

print("All filtered files saved.")

# Load filtered CSVs

In [4]:
import pandas as pd
business = pd.read_csv(DATA_DIR / "az_coffee_business.csv")
reviews = pd.read_csv(DATA_DIR / "az_coffee_reviews.csv")
tips = pd.read_csv(DATA_DIR / "az_coffee_tips.csv")
checkins = pd.read_csv(DATA_DIR / "az_coffee_checkins.csv")
users = pd.read_csv(DATA_DIR / "az_coffee_users.csv")

In [9]:
business.head()
business.info()
reviews.head()
reviews.info()
tips.head()
tips.info()
checkins.head()
checkins.info()
users.head()
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570 entries, 0 to 569
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   business_id   570 non-null    object 
 1   name          570 non-null    object 
 2   city          570 non-null    object 
 3   state         570 non-null    object 
 4   latitude      570 non-null    float64
 5   longitude     570 non-null    float64
 6   stars         570 non-null    float64
 7   review_count  570 non-null    int64  
 8   categories    570 non-null    object 
 9   is_open       570 non-null    int64  
dtypes: float64(3), int64(2), object(5)
memory usage: 44.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51307 entries, 0 to 51306
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   review_id      51307 non-null  object
 1   user_id        51307 non-null  object
 2   business_id    51307 non-null 

# Basic cleaning

In [9]:
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")
tips["date"] = pd.to_datetime(tips["date"], errors="coerce")

reviews["review_length"] = reviews["text"].fillna("").str.len()
tips["tip_length"] = tips["text"].fillna("").str.len()

checkins["checkin_dates"] = checkins["date"].fillna("")
checkins["total_checkins"] = checkins["checkin_dates"].apply(
    lambda x: 0 if not x else len([d for d in str(x).split(",") if d.strip()])
)

# Load into SQLite

In [10]:
conn = sqlite3.connect("../coffee_analysis.db")

business.to_sql("business", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)
tips.to_sql("tips", conn, if_exists="replace", index=False)
checkins.to_sql("checkins", conn, if_exists="replace", index=False)
users.to_sql("users", conn, if_exists="replace", index=False)

print("SQLite database created.")

SQLite database created.
